Vocabulary & sequence encoding for steps_tokens

Reads recipes_final.csv, inserts <EOI> boundary tokens at each period in steps_tokens, builds a word-to-index vocabulary (min-frequency cutoff), converts each recipe to a padded index sequence, and trains a Word2Vec embedding matrix on this corpus for RNN_enc/RNN_dec.

In [ ]:
import ast
import json
from collections import Counter

import numpy as np
import pandas as pd
from gensim.models import Word2Vec

In [ ]:
df = pd.read_csv('./data/recipes_final.csv', index_col=0)
df['steps_tokens'] = df['steps_tokens'].apply(ast.literal_eval)
df[['name', 'cuisine', 'steps_tokens']].head()

Instruction boundaries (<EOI>)

steps_tokens still has literal '.' tokens marking instruction ends (punctuation was kept for this column). Replace each '.' with an <EOI> token so the decoder gets an explicit per-instruction boundary marker instead of an ordinary punctuation character.

In [ ]:
EOI = '<EOI>'

def insert_eoi(tokens):
    return [EOI if t == '.' else t for t in tokens]

df['steps_tokens_eoi'] = df['steps_tokens'].apply(insert_eoi)
df['steps_tokens_eoi'].iloc[0]

Vocabulary (min-frequency cutoff)

Drop words appearing fewer than MIN_FREQ times across the corpus to control vocab size, then add PAD/UNK/SOS/EOS on top of it.

In [ ]:
PAD, UNK, SOS, EOS = '<PAD>', '<UNK>', '<SOS>', '<EOS>'
SPECIAL_TOKENS = [PAD, UNK, SOS, EOS]
MIN_FREQ = 3

token_counts = Counter()
for tokens in df['steps_tokens_eoi']:
    token_counts.update(tokens)

vocab_words = sorted(w for w, c in token_counts.items() if c >= MIN_FREQ)

word_to_idx = {tok: i for i, tok in enumerate(SPECIAL_TOKENS)}
for w in vocab_words:
    word_to_idx[w] = len(word_to_idx)

idx_to_word = {i: w for w, i in word_to_idx.items()}
VOCAB_SIZE = len(word_to_idx)

print(f"raw token types: {len(token_counts)}")
print(f"vocab after min_freq={MIN_FREQ} cutoff (+specials): {VOCAB_SIZE}")

Encode to index sequences (<SOS> ... <EOS>, OOV -> <UNK>)

In [ ]:
unk_idx, sos_idx, eos_idx = word_to_idx[UNK], word_to_idx[SOS], word_to_idx[EOS]

def encode_tokens(tokens):
    return [sos_idx] + [word_to_idx.get(t, unk_idx) for t in tokens] + [eos_idx]

df['steps_ids'] = df['steps_tokens_eoi'].apply(encode_tokens)
df['steps_ids'].iloc[0]

Sequence length distribution & max_len

Length here includes <SOS>/<EOS>/<EOI>, so it will run longer than the earlier number_of_tokens column from the data-prep notebook.

In [ ]:
df['sequence_length'] = df['steps_ids'].apply(len)
print(df['sequence_length'].describe())

MAX_LEN = int(df['sequence_length'].quantile(0.95))
truncated_share = (df['sequence_length'] > MAX_LEN).mean()
print(f"\nmax_len (95th percentile): {MAX_LEN}")
print(f"recipes truncated at this max_len: {truncated_share:.2%}")

Pad / truncate to max_len

Truncation keeps <SOS> and the sequence's own final token (<EOS>) and drops from the end of the instruction body, so every row still ends on a valid <EOS>.

In [ ]:
pad_idx = word_to_idx[PAD]

def pad_or_truncate(ids):
    if len(ids) >= MAX_LEN:
        return ids[:MAX_LEN - 1] + [ids[-1]]
    return ids + [pad_idx] * (MAX_LEN - len(ids))

df['steps_ids_padded'] = df['steps_ids'].apply(pad_or_truncate)

sequence_tensor = np.array(df['steps_ids_padded'].tolist(), dtype=np.int64)
sequence_tensor.shape

Word2Vec embeddings, trained on this corpus

Trained on the raw token lists (with <EOI>, without the synthetic PAD/UNK/SOS/EOS added only for the tensor step) so every real word's vector reflects actual recipe-instruction co-occurrence. Embeddings are fine-tuned later during autoencoder training, so this is just the initialization.

In [ ]:
EMBED_DIM = 300

w2v_model = Word2Vec(
    sentences=df['steps_tokens_eoi'].tolist(),
    vector_size=EMBED_DIM,
    window=5,
    min_count=MIN_FREQ,
    workers=4,
    seed=42,
)
len(w2v_model.wv)

Embedding matrix aligned to vocab indices (random init where Word2Vec has no vector, e.g. PAD/UNK/SOS/EOS or anything below MIN_FREQ in Word2Vec's own count)

In [ ]:
rng = np.random.default_rng(42)
embedding_matrix = rng.normal(scale=0.1, size=(VOCAB_SIZE, EMBED_DIM)).astype(np.float32)
embedding_matrix[pad_idx] = 0.0

hits = 0
for word, idx in word_to_idx.items():
    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]
        hits += 1

print(f"{hits}/{VOCAB_SIZE} vocab entries initialized from Word2Vec ({hits / VOCAB_SIZE:.1%})")

Persist vocab, embeddings, and encoded sequences for the training script

In [ ]:
np.save('./data/steps_sequence_tensor.npy', sequence_tensor)
np.save('./data/steps_embedding_matrix.npy', embedding_matrix)

with open('./data/steps_vocab.json', 'w') as f:
    json.dump(
        {'word_to_idx': word_to_idx, 'max_len': MAX_LEN, 'min_freq': MIN_FREQ, 'embed_dim': EMBED_DIM},
        f,
    )

print('saved: steps_sequence_tensor.npy', sequence_tensor.shape)
print('saved: steps_embedding_matrix.npy', embedding_matrix.shape)
print('saved: steps_vocab.json, vocab_size =', VOCAB_SIZE)